# Structured Output - JSON & Response Schema

## Why Do We Need Structured Output?

In our previous chatbots, the LLM returns the response as **plain text**.

Plain text is easy for humans to read, but it is difficult for an application to directly use, store, or process.

For example, when the same question is asked by multiple users, the LLM may generate different responses even when the meaning is the same. Storing all these free-form responses in a database is inefficient because the data does not follow a fixed structure.

Our application, however, needs data in a **defined structure**

So, there is a gap between:

**LLM → plain text** and **Application → structured data**

### Solution: Structured Output

To solve this problem, we can instruct the LLM to return data in a **predefined structure** instead of a paragraph.

This is called **structured output**.

For our example, we use **JSON (JavaScript Object Notation)** as the structured format because it is widely supported by programming languages, APIs, databases, and frameworks.

So the flow becomes:

**Unstructured LLM response → Structured JSON → Application**

The goal is to extract information from a paragraph and return it as structured JSON data.

---

## Python and JSON

Since our code is written in Python, we need to understand how Python data types correspond to JSON data types.

| Python                  | JSON             |
| ----------------------- | ---------------- |
| `True` / `False`        | `true` / `false` |
| `None`                  | `null`           |
| `'string'` / `"string"` | `"string"`       |

Python allows strings to be enclosed in either single or double quotes, whereas JSON strings must use **double quotes**.

### Common JSON Data Types

* String
* Number
* Boolean
* Array
* Object

### `json.loads()`

Converts a JSON string into a Python object.

```python
json.loads()
```

### `json.dumps()`

Converts a Python object into a JSON-formatted string.

```python
json.dumps()
```


##Ways to Generate JSON Output

There are three approaches:

## 1.Asking for JSON through the prompt

## 2.Using response_mime_type

## 3.Using response_schema

In [2]:
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata

client=genai.Client(api_key=userdata.get('Ragproject'))


prompt="""

Return the details from the LinkedIn post in JSON format.Make sure the JSON contains all five keys:role, company,
stipend, location, duration. Do not hallucinate or infer any values. If any information is not available in the LinkedIn post,
set its value to null.

post=We are hiring Frontend Developer Interns at Zoho in Madurai.
This is a 9-month internship program with a stipend of ₹15,000 per month.
Candidates interested in gaining practical experience in frontend development can apply before 31st August. """


In [6]:
# 1.Asking for JSON through the prompt

response_1=client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt,
)

print(response_1.text)

```json
{
  "role": "Frontend Developer Intern",
  "company": "Zoho",
  "stipend": "₹15,000 per month",
  "location": "Madurai",
  "duration": "9-month"
}
```


In [10]:
#The model may return the JSON inside Markdown code fences (```json { }```)
#The code fences are Markdown formatting, not part of valid JSON. Therefore, passing `response_1.text` directly to `json.loads()` can result in a JSON decoding error.

# 2.Using response_mime_type

import json

response_2=client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type='application/json'
    )
)

#print(response_2.text) this also gives the same json format response

data_2=json.loads(response_2.text)
print(json.dumps(data_2, indent=2))

{
  "role": "Frontend Developer Intern",
  "company": "Zoho",
  "stipend": "\u20b915,000 per month",
  "location": "Madurai",
  "duration": "9-month"
}


In [19]:
#Although the output is valid JSON, the required **data types and structure** are not explicitly defined.
#example:"stipend": "₹15,000 per month" the value is a string instead of an integer, which can cause a data-type mismatch when inserting it into a database.

#3.Using response_schema

from pydantic import BaseModel  #pydantic- lib for data vallidataion, basemodel-class we use to define our structure

class internship(BaseModel):  #internship- is our schema
  role: str
  company: str
  stipend: int
  location: str
  duration: str

response_3=client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=internship
    )
)

data_3=json.loads(response_3.text)  #holds pytho dict format

expected_keys=['role','company','stipend','location','duration']  #ensure that evry expected key exist in dict
for key in expected_keys:
  data_3.setdefault(key,None) #if a key is missing, it is added with a value none

print("Json Formet")
print(json.dumps(data_3,indent=2))

print()

print("Accessing from dictionar")
print("Role: ",data_3["role"])
print("Company: ",data_3["company"])
print("Stipend: ",data_3["stipend"])
print("Location: ",data_3["location"])
print("Duration: ",data_3["duration"])

Json Formet
{
  "role": "Frontend Developer Intern",
  "company": "Zoho",
  "stipend": 15000,
  "location": "Madurai",
  "duration": "9-month"
}

Accessing from dictionar
Role:  Frontend Developer Intern
Company:  Zoho
Stipend:  15000
Location:  Madurai
Duration:  9-month


Plain text- difficult for application to process

Structured json - predictable and machine-readable

response_mime_type - specific json structure and types

response_schema - specific json structure and data types

Therefore, structured output brings the gap between natural language LLM responses and application-ready data.